In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config

Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
%pip install pytrends
print("ready")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
ready


In [0]:
dbutils.library.restartPython()

In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config

Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 38


In [0]:
from pytrends.request import TrendReq
from datetime import datetime, timedelta
import pandas as pd
import time

pytrends = TrendReq(hl='en-US', tz=360)

results = []
failed = []

for film in FILM_LIST:
    title = film["title"]
    release_date = datetime.strptime(film["release_date"], "%Y-%m-%d")
    
    # 90 days AFTER release
    start = release_date.strftime("%Y-%m-%d")
    end = (release_date + timedelta(days=90)).strftime("%Y-%m-%d")
    
    # Don't pull future dates
    today = datetime.today()
    if release_date > today:
        print(f"⏭️ {title} — future release, skipping")
        continue
    if datetime.strptime(end, "%Y-%m-%d") > today:
        end = today.strftime("%Y-%m-%d")
    
    timeframe = f"{start} {end}"
    
    success = False
    attempts = 0
    
    while not success and attempts < 3:
        try:
            pytrends.build_payload([title], timeframe=timeframe, geo='US')
            df = pytrends.interest_over_time()
            
            if not df.empty:
                df = df.reset_index()
                df["film"] = title
                df["release_date"] = film["release_date"]
                df["genre"] = film["genre"]
                df = df.rename(columns={title: "interest_score"})
                df = df[["date", "film", "release_date", 
                          "genre", "interest_score"]]
                results.append(df)
                print(f"✅ {title} — {len(df)} rows")
                success = True
            else:
                print(f"⚠️ {title} — no data")
                success = True
                
        except Exception as e:
            attempts += 1
            if "429" in str(e):
                print(f"⏳ {title} — rate limited, waiting 60s (attempt {attempts}/3)")
                time.sleep(60)
            else:
                print(f"❌ {title} — {e}")
                failed.append(title)
                success = True
    
    if not success:
        failed.append(title)
    
    time.sleep(15)

if results:
    df_post = pd.concat(results, ignore_index=True)
    print(f"\n✅ Total rows: {len(df_post)}")
    if failed:
        print(f"❌ Failed: {failed}")
    display(df_post.head(10))
else:
    print("No data collected.")

✅ Batman Begins — 91 rows
✅ Brokeback Mountain — 91 rows
✅ Casino Royale — 91 rows
✅ The Departed — 91 rows
✅ No Country for Old Men — 91 rows
✅ There Will Be Blood — 91 rows
✅ Ratatouille — 91 rows
✅ The Dark Knight — 91 rows
✅ Inception — 91 rows
✅ Avatar — 91 rows
✅ The Social Network — 91 rows
✅ Interstellar — 91 rows
✅ Gone Girl — 91 rows
✅ The Wolf of Wall Street — 91 rows
✅ Gravity — 91 rows
✅ The Hangover — 91 rows
✅ Bridesmaids — 91 rows
✅ Paranormal Activity — 91 rows
✅ The Conjuring — 91 rows
✅ Mad Max: Fury Road — 91 rows
✅ The Revenant — 91 rows
✅ La La Land — 91 rows
✅ Get Out — 91 rows
✅ It — 91 rows
✅ Black Panther — 91 rows
✅ Crazy Rich Asians — 91 rows
✅ A Quiet Place — 91 rows
✅ Hereditary — 91 rows
✅ Bohemian Rhapsody — 91 rows
✅ Joker — 91 rows
✅ Parasite — 91 rows
✅ Tenet — 91 rows
✅ Dune — 91 rows
✅ The Batman — 91 rows
✅ Top Gun: Maverick — 91 rows
✅ Everything Everywhere All at Once — 91 rows
✅ Morbius — 91 rows
✅ Nope — 91 rows
✅ The Menu — 91 rows
✅ Oppenheim

date,film,release_date,genre,interest_score
2005-06-15T00:00:00.000Z,Batman Begins,2005-06-15,Action,71
2005-06-16T00:00:00.000Z,Batman Begins,2005-06-15,Action,85
2005-06-17T00:00:00.000Z,Batman Begins,2005-06-15,Action,74
2005-06-18T00:00:00.000Z,Batman Begins,2005-06-15,Action,92
2005-06-19T00:00:00.000Z,Batman Begins,2005-06-15,Action,100
2005-06-20T00:00:00.000Z,Batman Begins,2005-06-15,Action,68
2005-06-21T00:00:00.000Z,Batman Begins,2005-06-15,Action,45
2005-06-22T00:00:00.000Z,Batman Begins,2005-06-15,Action,38
2005-06-23T00:00:00.000Z,Batman Begins,2005-06-15,Action,32
2005-06-24T00:00:00.000Z,Batman Begins,2005-06-15,Action,34


In [0]:
df_spark = spark.createDataFrame(df_post)

(df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE_PATH}.google_trends_postrelease_raw")
)

print(f"✅ Saved to {BRONZE_PATH}.google_trends_postrelease_raw")
print(f"   Rows written: {df_spark.count()}")

✅ Saved to bootcamp_students.tiffani_ceresrain_bronze.google_trends_postrelease_raw
   Rows written: 4550
